# Lumina LOB — Python vs C++ Engine Benchmark Report

This notebook measures and visualises the throughput of the two matching-engine implementations shipped with **Lumina LOB**:

1. **Pure-Python engine** (`lumina_lob.core.MatchingEngine`)
2. **C++17 extension** (`lumina_lob._core.MatchingEngine`) exposed via pybind11

The benchmark submits identical deterministic streams of limit orders to both engines and reports events per second.

> **CI guard:** set the environment variable `LUMINA_BENCHMARK_QUICK=1` to run a tiny workload (1k and 5k orders) instead of the default suite.

## Methodology

- Order stream: alternating bid/ask limit orders with random prices around a central price (`10_000 ± 2` ticks) and random quantities (`1–100`).
- Each engine receives the **same** stream for a given seed and order count.
- Throughput is computed as `orders / elapsed_wall_time`.
- The C++ path still crosses the pybind11 boundary once per order; this is the dominant bottleneck at small order sizes. A future batch-submission API can remove that per-call overhead and push the C++ engine toward its raw speed.

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BENCHMARK_MODULE = "benchmarks.engine_benchmark"


def run_benchmark(orders: int, seed: int = 42) -> dict[str, float | None]:
    """Run the benchmark script and parse the reported rates."""
    cmd = [
        sys.executable,
        "-m",
        BENCHMARK_MODULE,
        "--orders",
        str(orders),
        "--seed",
        str(seed),
    ]
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=True,
        cwd=REPO_ROOT,
    )
    output = result.stdout

    def _parse(label: str) -> float | None:
        pattern = rf"{label}:\s+([0-9,.]+|not built)\s+events/sec"
        match = re.search(pattern, output)
        if not match:
            return None
        value = match.group(1).replace(",", "")
        if value == "not built":
            return None
        return float(value)

    return {
        "orders": orders,
        "python_eps": _parse("Python engine"),
        "cpp_eps": _parse("C\+\+ engine"),
    }


print("Benchmark helper ready.")

In [ ]:
# CI guard: tiny workload when LUMINA_BENCHMARK_QUICK is set
if os.environ.get("LUMINA_BENCHMARK_QUICK"):
    ORDER_COUNTS = [1_000, 5_000]
else:
    ORDER_COUNTS = [10_000, 50_000, 100_000, 200_000]

SEED = 42
print(f"Running benchmark for order counts: {ORDER_COUNTS}")

In [ ]:
records = [run_benchmark(n, SEED) for n in ORDER_COUNTS]
df = pd.DataFrame(records)
df["speedup"] = df["cpp_eps"] / df["python_eps"]
df = df.round({"python_eps": 1, "cpp_eps": 1, "speedup": 2})
display(df)

## Results table

The table above shows throughput (events/sec) and the C++ speedup factor for each workload size.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
width = 0.35
x = range(len(df))

ax.bar([i - width / 2 for i in x], df["python_eps"], width, label="Python")
ax.bar([i + width / 2 for i in x], df["cpp_eps"], width, label="C++")
ax.set_xlabel("Orders submitted")
ax.set_ylabel("Events / sec")
ax.set_title("Matching-engine throughput by workload size")
ax.set_xticks(x)
ax.set_xticklabels(df["orders"])
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(df["orders"], df["speedup"], marker="o")
ax.set_xlabel("Orders submitted")
ax.set_ylabel("C++ speedup (x)")
ax.set_title("C++ speedup vs Python engine")
ax.grid(linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## Interpretation and next steps

- The pure-Python engine is already fast enough for backtesting and RL training where each step does significant work.
- The C++ extension shows a measurable but modest speedup in this per-order pybind11 configuration because every order pays the Python ↔ C++ crossing cost.
- To hit the project target of **>10M events/sec** for the C++ engine, the next performance step is a **batch-submission API** (`engine.process_batch(orders)`) that passes a list of order tuples to C++ in a single call and runs the matching loop entirely in native code.
- That batch API is intentionally left for a future optimisation pass so the current checkpoint (CP4.5) stays focused on measurement and reporting.